<a href="https://colab.research.google.com/github/mihnea-popescu/yolov1-cupy/blob/linear_class/notebooks/linear.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!git clone https://github.com/mihnea-popescu/yolov1-cupy.git
%cd yolov1-cupy
!git checkout linear_class

Cloning into 'yolov1-cupy'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 18 (delta 2), reused 18 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 5.74 KiB | 5.74 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/yolov1-cupy/yolov1-cupy
Branch 'linear_class' set up to track remote branch 'linear_class' from 'origin'.
Switched to a new branch 'linear_class'


In [3]:
import numpy as np
from linear import Linear

Test 1: Output shape is correct

In [5]:
layer = Linear(in_features=8, out_features=4)
x = np.random.randn(16, 8)   # batch of 16, 8 features each
out = layer(x)
assert out.shape == (16, 4), f"Expected (16, 4), got {out.shape}"
print("Test 1 passed: output shape correct")

Test 1 passed: output shape correct


Test 2: Matches PyTorch output numerically

In [6]:
import torch
import torch.nn as nn

# Build our layer and a PyTorch layer with IDENTICAL weights
our_layer = Linear(in_features=4, out_features=3, bias=True)
torch_layer = nn.Linear(4, 3)

# Copy our weights into PyTorch layer
torch_layer.weight.data = torch.tensor(our_layer.W, dtype=torch.float32)
torch_layer.bias.data   = torch.tensor(our_layer.b, dtype=torch.float32)

x_np = np.random.randn(5, 4)
our_out    = our_layer(x_np)
torch_out  = torch_layer(torch.tensor(x_np, dtype=torch.float32)).detach().numpy()

np.testing.assert_allclose(our_out, torch_out, rtol=1e-5)
print("Test 2 passed: matches PyTorch output numerically")

Test 2 passed: matches PyTorch output numerically


Test 3: Backward pass gradient shape

In [7]:
layer = Linear(4, 3)
x = np.random.randn(5, 4)
out = layer(x)

# Simulate a gradient flowing back (e.g., all ones)
d_out = np.ones_like(out)
d_input = layer.backward(d_out)

assert d_input.shape == x.shape,   f"d_input shape mismatch: {d_input.shape}"
assert layer.dW.shape == layer.W.shape, "dW shape mismatch"
assert layer.db.shape == layer.b.shape, "db shape mismatch"
print("Test 3 passed: backward gradients have correct shapes")

Test 3 passed: backward gradients have correct shapes


Test 4: No-bias mode works

In [8]:
layer = Linear(4, 3, bias=False)
x = np.random.randn(5, 4)
out = layer(x)
assert out.shape == (5, 3)
print("Test 4 passed: bias=False works correctly")

Test 4 passed: bias=False works correctly
